In [1]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy import stats

In [5]:
rng_test = np.random.default_rng(seed=0)

# Валидация алгоритма Швайнхарта

>Фрактальная размерность куба в 8D ($d^* = 3$):

In [6]:
N = 100_000

cube_3d = rng_test.uniform(0, 1, size=(N, 3))
noise = rng_test.normal(0, 0.001, size=(N, 5))
cube_8d = np.hstack([cube_3d, noise])

In [3]:
alpha_values = np.arange(1.0, 8.1, 0.2)
size_grid = np.unique(np.round(np.logspace(np.log10(200), np.log10(2000), 10)).astype(int))
n_repeats = 5

def mst_edge_weights(points):
    dist_matrix = cdist(points, points, metric='euclidean')
    mst = minimum_spanning_tree(dist_matrix)
    return mst.data

def compute_E_alpha(edge_weights, alpha_values):
    return np.array([np.sum(edge_weights ** alpha) for alpha in alpha_values])

In [13]:
E_matrix = np.zeros((len(alpha_values), len(size_grid)))

for j, size in enumerate(size_grid):
    E_per_repeat = np.zeros((n_repeats, len(alpha_values)))
    for r in range(n_repeats):
        idx = rng_test.choice(N, size=size, replace=False)
        pts = cube_8d[idx]
        edge_weights = mst_edge_weights(pts)
        E_per_repeat[r] = compute_E_alpha(edge_weights, alpha_values)
    E_matrix[:, j] = E_per_repeat.mean(axis=0)

In [5]:
def fit_regression(size_grid, E_values, alpha):
    ln_size = np.log(size_grid.astype(float))
    ln_E = np.log(E_values)
    slope, intercept, r, p, se_slope = stats.linregress(ln_size, ln_E)
    beta = slope
    if beta <= 0 or beta >= 1:
        return dict(d_hat=np.nan, beta=beta, valid=False)
    d_hat = alpha / (1.0 - beta)
    n = len(ln_size)
    t_crit = stats.t.ppf(0.975, df=n - 2)
    ci_beta = t_crit * se_slope
    gamma = 0.10
    rel_ci = ci_beta / (abs(beta) + 1e-9)
    valid = rel_ci <= gamma
    return dict(d_hat=d_hat, beta=beta, rel_ci=rel_ci, valid=valid)

In [14]:
valid_d_hats = []
for a_idx, alpha in enumerate(alpha_values):
    E_values = E_matrix[a_idx, :]
    mask = (E_values > 0) & np.isfinite(E_values)
    if mask.sum() < 3:
        continue
    res = fit_regression(size_grid[mask], E_values[mask], alpha)
    if res['valid'] and np.isfinite(res['d_hat']) and res['d_hat'] > 0:
        valid_d_hats.append(res['d_hat'])

In [17]:
if valid_d_hats:
    print(f"frct dim: min={np.min(valid_d_hats):.3f}, max={np.max(valid_d_hats):.3f}, median={np.median(valid_d_hats):.3f}")
else:
    print("Нет валидных оценок!")

frct dim: min=2.846, max=2.904, median=2.870


> Фрактальная размерность поверхность сферы в 8D ($d^* = 2$)

In [18]:
points = rng_test.normal(0, 1, size=(N, 3))
points /= np.linalg.norm(points, axis=1, keepdims=True)
sphere_8d = np.hstack([points, np.zeros((N, 5))])

In [19]:
E_matrix = np.zeros((len(alpha_values), len(size_grid)))

for j, size in enumerate(size_grid):
    E_per_repeat = np.zeros((n_repeats, len(alpha_values)))
    for r in range(n_repeats):
        idx = rng_test.choice(N, size=size, replace=False)
        pts = sphere_8d[idx]
        edge_weights = mst_edge_weights(pts)
        E_per_repeat[r] = compute_E_alpha(edge_weights, alpha_values)
    E_matrix[:, j] = E_per_repeat.mean(axis=0)

In [20]:
valid_d_hats = []
for a_idx, alpha in enumerate(alpha_values):
    E_values = E_matrix[a_idx, :]
    mask = (E_values > 0) & np.isfinite(E_values)
    if mask.sum() < 3:
        continue
    res = fit_regression(size_grid[mask], E_values[mask], alpha)
    if res['valid'] and np.isfinite(res['d_hat']) and res['d_hat'] > 0:
        valid_d_hats.append(res['d_hat'])

In [21]:
if valid_d_hats:
    print(f"frct dim: min={np.min(valid_d_hats):.3f}, max={np.max(valid_d_hats):.3f}, median={np.median(valid_d_hats):.3f}")
else:
    print("Нет валидных оценок!")

frct dim: min=2.014, max=2.022, median=2.017


> Фрактальная размерноть шара в 8D ($d^* = 3$):

In [ ]:
N = 100_000
batch = 10_000
points = []

while sum(len(p) for p in points) < N:
    p = rng_test.uniform(-1, 1, size=(batch, 3))
    mask = np.linalg.norm(p, axis=1) <= 1.0
    points.append(p[mask])

ball_3d = np.vstack(points)[:N]
ball_8d = np.hstack([ball_3d, np.zeros((N, 5))])

In [24]:
E_matrix = np.zeros((len(alpha_values), len(size_grid)))

for j, size in enumerate(size_grid):
    E_per_repeat = np.zeros((n_repeats, len(alpha_values)))
    for r in range(n_repeats):
        idx = rng_test.choice(N, size=size, replace=False)
        pts = ball_8d[idx]
        edge_weights = mst_edge_weights(pts)
        E_per_repeat[r] = compute_E_alpha(edge_weights, alpha_values)
    E_matrix[:, j] = E_per_repeat.mean(axis=0)

In [25]:
valid_d_hats = []
for a_idx, alpha in enumerate(alpha_values):
    E_values = E_matrix[a_idx, :]
    mask = (E_values > 0) & np.isfinite(E_values)
    if mask.sum() < 3:
        continue
    res = fit_regression(size_grid[mask], E_values[mask], alpha)
    if res['valid'] and np.isfinite(res['d_hat']) and res['d_hat'] > 0:
        valid_d_hats.append(res['d_hat'])

In [26]:
if valid_d_hats:
    print(f"frct dim: min={np.min(valid_d_hats):.3f}, max={np.max(valid_d_hats):.3f}, median={np.median(valid_d_hats):.3f}")
else:
    print("Нет валидных оценок!")

frct dim: min=2.863, max=2.870, median=2.866


> Фрактальная размерность ковра Серпинского ($d^* = 1.585$)

In [27]:
N = 100_000
warmup = 100

point = np.array([0.0, 0.0])
samples = np.zeros((N, 2))
vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3)/2]])

for i in range(-warmup, N):
    v = vertices[rng_test.integers(0, 3)]
    point = 0.5 * (point + v)
    if i >= 0:
        samples[i] = point

sierpinski_8d = np.hstack([samples, np.zeros((N, 6))])

In [28]:
E_matrix = np.zeros((len(alpha_values), len(size_grid)))

for j, size in enumerate(size_grid):
    E_per_repeat = np.zeros((n_repeats, len(alpha_values)))
    for r in range(n_repeats):
        idx = rng_test.choice(N, size=size, replace=False)
        pts = sierpinski_8d[idx]
        edge_weights = mst_edge_weights(pts)
        E_per_repeat[r] = compute_E_alpha(edge_weights, alpha_values)
    E_matrix[:, j] = E_per_repeat.mean(axis=0)

In [29]:
valid_d_hats = []
for a_idx, alpha in enumerate(alpha_values):
    E_values = E_matrix[a_idx, :]
    mask = (E_values > 0) & np.isfinite(E_values)
    if mask.sum() < 3:
        continue
    res = fit_regression(size_grid[mask], E_values[mask], alpha)
    if res['valid'] and np.isfinite(res['d_hat']) and res['d_hat'] > 0:
        valid_d_hats.append(res['d_hat'])

In [30]:
if valid_d_hats:
    print(f"frct dim: min={np.min(valid_d_hats):.3f}, max={np.max(valid_d_hats):.3f}, median={np.median(valid_d_hats):.3f}")
else:
    print("Нет валидных оценок!")

frct dim: min=1.592, max=1.596, median=1.594


> Фрактальная размерность снежинки Коха ($d^* = 1.2619$)

In [36]:
def make_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

R0 = make_rotation(0)
R_plus = make_rotation(np.pi / 3)
R_minus = make_rotation(-np.pi / 3)

transforms = [(R0, np.array([0.0, 0.0])), (R_plus,  np.array([1/3, 0.0])), (R_minus, np.array([1/2, np.sqrt(3)/6])), (R0, np.array([2/3, 0.0]))]
N = 100_000
warmup = 100

point = np.array([0.0, 0.0])
samples = np.zeros((N, 2))

for i in range(-warmup, N):
    R, t = transforms[rng_test.integers(0, 4)]
    point = (1/3) * R @ point + t
    if i >= 0:
        samples[i] = point

koch_8d = np.hstack([samples, np.zeros((N, 6))])

In [37]:
E_matrix = np.zeros((len(alpha_values), len(size_grid)))

for j, size in enumerate(size_grid):
    E_per_repeat = np.zeros((n_repeats, len(alpha_values)))
    for r in range(n_repeats):
        idx = rng_test.choice(N, size=size, replace=False)
        pts = koch_8d[idx]
        edge_weights = mst_edge_weights(pts)
        E_per_repeat[r] = compute_E_alpha(edge_weights, alpha_values)
    E_matrix[:, j] = E_per_repeat.mean(axis=0)

In [38]:
valid_d_hats = []
for a_idx, alpha in enumerate(alpha_values):
    E_values = E_matrix[a_idx, :]
    mask = (E_values > 0) & np.isfinite(E_values)
    if mask.sum() < 3:
        continue
    res = fit_regression(size_grid[mask], E_values[mask], alpha)
    if res['valid'] and np.isfinite(res['d_hat']) and res['d_hat'] > 0:
        valid_d_hats.append(res['d_hat'])

In [39]:
if valid_d_hats:
    print(f"frct dim: min={np.min(valid_d_hats):.3f}, max={np.max(valid_d_hats):.3f}, median={np.median(valid_d_hats):.3f}")
else:
    print("Нет валидных оценок!")

frct dim: min=1.264, max=1.265, median=1.264


# Валидация алгоритма MLE

In [41]:
from scipy.spatial import KDTree

In [42]:
def mle_dim_from_points(points, k=10):
    tree = KDTree(points)
    distances, _ = tree.query(points, k=k + 1)
    dists = distances[:, 1:]

    T_k = dists[:, -1]
    T_j = dists[:, :-1]

    eps = 1e-10
    T_k_safe = np.maximum(T_k[:, np.newaxis], eps)
    T_j_safe = np.maximum(T_j, eps)

    log_ratios = np.log(T_k_safe / T_j_safe)
    mean_log = np.mean(log_ratios, axis=1)

    valid = mean_log > eps
    if valid.sum() == 0:
        return np.nan

    m_hat = 1.0 / mean_log[valid]
    return float(np.median(m_hat))

In [46]:
print(f"Куб: {mle_dim_from_points(cube_3d):.3f}")
print(f"Шар: {mle_dim_from_points(ball_3d):.3f}")
print(f"Поверхность сферы: {mle_dim_from_points(sphere_8d):.3f}")  
print(f"Серпинский: {mle_dim_from_points(sierpinski_8d):.3f}")
print(f"Кох: {mle_dim_from_points(koch_8d):.3f}")

Куб: 3.085
Шар: 3.081
Поверхность сферы: 2.077
Серпинский: 1.644
Кох: 1.311


# Валидация ``Энтропия-Сложность`` плоскости

In [2]:
import ordpy

In [3]:
def compute_HC(signal, D, tau=1):
    """
    Computing the (H, C) point on the entropy-complexity plane.
    
    signal- audionumpy array
    D - embedding dimension
    tau - embedding delay
    
    Returns:
        (H, C) — float values of entropy and complexity, or (None, None) if calculation failed
    """
    try:
        H, C = ordpy.complexity_entropy(signal, dx=D, taux=tau)
        return H, C
    except Exception as e:
        print(f"Error when computing HC for dx={D}, taux={tau}: {e}")
        return None, None

In [7]:
N = 100_000

constant = np.ones(N)

white_noise = rng_test.uniform(0, 1, size=N)
D = 5

H_const, C_const = compute_HC(constant, D=D, tau=1)
H_noise, C_noise = compute_HC(white_noise, D=D, tau=1)

print(f"Константный: H={H_const:.4f}, C={C_const:.4f}")
print(f"Белый шум: H={H_noise:.4f}, C={C_noise:.4f}")

Константный: H=-0.0000, C=-0.0000
Белый шум: H=0.9999, C=0.0002
